# 02. Qwen으로 Train 칸 판정 (A100, 1~2시간)

01과 똑같은 질문·기준을 Train 전체에 쓴다. 2,000칸마다 드라이브에 저장하므로, **런타임이 끊기면 0단계부터 다시 실행**하면 이어서 한다.

| 단계 | 하는 일 | 쓰는 함수 |
|---|---|---|
| 0 | 준비, Qwen 불러오기 | `judge_llm.load_judge` |
| 1 | 판정할 칸 | `cells.build_cells` |
| 2 | **Qwen 판정** (이어서 가능) | `cells.judge_cells` |
| 3 | 판정표 저장 + 증상별 분포표 | `cells.save_judged`, `cleaning.action_summary` |

결과: `judge/train/`(블록), `judge/train_judged.csv`.

## 0. 준비

In [ ]:
# Qwen3는 transformers 4.51 이상이 필요하다. 설치 후 '세션 다시 시작' 메시지가 나오면 다시 시작하고 위 셀부터 실행
!pip -q install -U "transformers>=4.51" accelerate

In [ ]:
# 드라이브 연결 + src 폴더를 import할 수 있게 경로 추가
from google.colab import drive
drive.mount("/content/drive")

import os, sys
PROJECT_ROOT = "/content/drive/MyDrive/증상인식_최종_프로젝트"   # 폴더를 다른 곳에 두었으면 여기만 바꾸기
os.environ["SYMPTOM_FINAL_ROOT"] = PROJECT_ROOT
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
# src/*.py를 고친 뒤에는 런타임 → 세션 다시 시작 후 이 셀부터 다시 실행

In [ ]:
import pandas as pd
import torch
from src import config
from src.data import load_labels
from src.cells import build_cells, judge_cells, save_judged
from src.cleaning import decide_actions, action_summary
from src.judge_llm import load_judge

jcfg = config.JudgeConfig()     # 01과 같은 설정이어야 한다
OUT = f"{config.RESULTS_DIR}/02_train"
os.makedirs(OUT, exist_ok=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(GPU 런타임으로 바꾸기)")
tok, llm = load_judge(jcfg.model_name, jcfg.load_in_4bit)

## 1. 판정할 칸

In [ ]:
tr_df, _ = load_labels()
tr_cells = build_cells(tr_df)

## 2. Qwen 판정 (끊기면 0단계부터 다시)

In [ ]:
tr_probs, done = judge_cells(tr_cells, tok, llm, f"{config.JUDGE_DIR}/train", jcfg)

## 3. 판정표 저장 + 요약

In [ ]:
assert done, "판정이 덜 끝남. 위 셀을 다시 실행"
judged = save_judged(tr_cells, tr_probs, f"{config.JUDGE_DIR}/train_judged.csv")
summ = action_summary(decide_actions(judged, jcfg), tr_df, jcfg.warn_min_pos_train, jcfg)
summ.to_csv(f"{OUT}/처리_분포표.csv", encoding="utf-8-sig")
display(summ)